In [0]:
#Parametros de carga
dbutils.widgets.text("tipo_carga","Incremental")
dbutils.widgets.text("fecha_carga", "2025-06-30")

tipo_carga = dbutils.widgets.get("tipo_carga")
fecha_carga =  dbutils.widgets.get("fecha_carga")

In [0]:
from pyspark.sql.functions import col, to_date, when, lit, trim, lower, upper,substring_index, datediff, try_to_date, trunc, current_timestamp,row_number, initcap, lpad,substring, length, concat, split
from pyspark.sql.window import Window

In [0]:
df_compras_import = spark.table("linio.bronze_compras").select(col("venta_id"), col("factura"), col("fecha_orden"),
col("fecha_entrega"), col("fecha_envio"), col("estado"), col("cliente_code"), col("tipo_cliente"), col("nombres"), col("apellidos"),
col("vendedor"), col("departamento"), col("metodo_pago"), col("tipo_compra"))

In [0]:
df_enteros = df_compras_import.withColumn("venta_id", col("venta_id").cast("integer"))

In [0]:
df_fecha_limpia = df_enteros.withColumn("fecha_envio_limpia", when(trim(col("fecha_envio")) == '"NaN"', None).otherwise(col("fecha_envio")))
df_eliminando_fecha = df_fecha_limpia.drop("fecha_envio").withColumnRenamed("fecha_envio_limpia", "fecha_envio")

In [0]:
#df_enteros.select(col("fecha_entrega")).groupBy(col("fecha_entrega")).count().display()
#df_fecha_limpia.select(col("fecha_envio_limpia")).groupBy(col("fecha_envio_limpia")).count().display()
#df_compras_import.select(col("fecha_orden")).groupBy(col("fecha_orden")).count().display()

In [0]:
df_fechas = (
    df_eliminando_fecha.withColumn("fecha_orden", try_to_date(col("fecha_orden"),  "yyyy-MM-dd"))
                     .withColumn("fecha_entrega", try_to_date(col("fecha_entrega"), "dd/MM/yyyy"))
                     .withColumn("fecha_envio", try_to_date(col("fecha_envio"), "dd-MM-yy"))
)


In [0]:
df_quitar_espacios = (
                        df_fechas.withColumn("factura",upper(trim(col("factura"))))
                                .withColumn("vendedor", trim(col("vendedor")))
                                .withColumn("departamento", trim(col("departamento")))
                                .withColumn("metodo_pago", trim(col("metodo_pago")))

)

df_compras_clean = df_quitar_espacios

In [0]:
df_compras_transform = (
    df_compras_clean.withColumn("cliente_id", substring_index(col("cliente_code"),"-",1).cast("integer"))
                    .withColumn("num_documento", substring_index(col("cliente_code"), "-", -1).cast("string"))
                    .withColumn("vendedor", when(col("vendedor").isNull(), lit("No Identificado")).otherwise(col("vendedor")))
                    .withColumn("dias_envio", when(col("estado") == 5, datediff(col("fecha_envio"), col("fecha_orden"))).otherwise(None))
                    .withColumn("grupo_dias_envio", when(col("dias_envio").isNull(), None)
                                .when(col("dias_envio") <=3, lit("[0 - 3 días]"))
                                .when(col("dias_envio") <=7, lit("[4 - 7 días]"))
                                .when(col("dias_envio") >=8, lit("[más de 8 días]"))
                                .otherwise(None))
)

In [0]:
df_seleccionar_col = df_compras_transform.select(col("venta_id"), col("factura"), col("tipo_compra"), col("fecha_orden"), col("fecha_entrega"), col("fecha_envio"), col("estado"), col("cliente_id"), col("vendedor"), col("departamento"), col("metodo_pago"), col("dias_envio"), col("grupo_dias_envio"))

In [0]:
df_primer_dia_mes = df_seleccionar_col.withColumn("periodo", trunc(col("fecha_orden"), "month")).withColumn("fecha_actualizacion", current_timestamp())
df_compras = df_primer_dia_mes

In [0]:
#df_compras.write.mode("overwrite").format("delta").partitionBy("periodo").saveAsTable("linio.silver_compras")


## Creación del df_clientes

In [0]:
win = Window.partitionBy(col("cliente_id")).orderBy(col("fecha_orden").desc())

df_clientes_transform = df_compras_transform.withColumn("rank", row_number().over(win)).filter(col("rank") == 1).drop(col("rank"))

In [0]:
df_clientes_transform =(
    df_clientes_transform.select(
    col("cliente_id"), 
    lpad(trim(col("num_documento")),8,"0").alias("num_documento") , 
    trim(initcap(col("nombres"))).alias("nombres"), 
    trim(initcap(col("apellidos"))).alias("apellidos"),
    upper(trim(col("tipo_cliente"))).alias("tipo_cliente")
    )
    .withColumn("tipo_documento",
                  when(length(col("num_documento")) == 8, lit("DNI"))
                  .when((length(col("num_documento")) == 11) & (substring(col("num_documento"), 1, 2) == 10), lit("RUC10"))
                  .when((length(col("num_documento")) == 11) & (substring(col("num_documento"), 1, 2) == 20), lit("RUC2O"))
                  .otherwise(None)
                )
    .withColumn("nombre_completo", concat(col("nombres"), lit(" "), col("apellidos")))
)
    



In [0]:
df_clientes = df_clientes_transform.select(col("cliente_id"), col("tipo_documento"), col("num_documento"), col("nombre_completo"), col("tipo_cliente"), (current_timestamp()).alias("fecha_actualizacion")).filter(col("cliente_id").isNotNull())

In [0]:
#df_clientes.write.mode("overwrite").format("delta").saveAsTable("linio.silver_clientes")

##Limpieza y transformación de la tabla bronze_detalles

In [0]:
df_detalles_import = spark.table("linio.bronze_detalles")

In [0]:
df_detalles_clean = df_detalles_import.withColumn("detalle_id", col("detalle_id").cast("integer")).withColumn("unidades", col("unidades").cast("integer")).withColumn("precio_unitario", col("precio_unitario").cast("double")).withColumn("factura", upper(trim(col("factura")))).withColumn("producto", trim(col("producto")))

In [0]:
df_detalles_transform = df_detalles_clean.withColumn("Subtotal", (col("unidades") * col("precio_unitario"))).withColumn("tienda", split(col("nombre_archivo"),"\\.")[0])

In [0]:
win = Window.partitionBy(col("producto")).orderBy(col("producto"))
df_productos_filtro = df_detalles_transform.withColumn("rank", row_number().over(win)).filter(col("rank") == 1)

In [0]:
df_productos = df_productos_filtro.select(trim(col("producto")).alias("producto"), upper(trim(col("categoria"))).alias("categoria"), upper(trim(col("subcategoria"))).alias("subcategoria"),  current_timestamp().alias("fecha_actualizacion"))

In [0]:
#df_productos.write.mode("overwrite").format("delta").saveAsTable("linio.silver_productos")

## Creación del dataframe “df_detalles”

In [0]:
df_productos_new = spark.table("linio.silver_productos").select(col("producto_id"), col("producto"))

In [0]:
df_detalles = (
    df_detalles_transform.alias("det")
    .join(df_productos_new.alias("prod"), "producto", "inner")
    .select(
        col("det.detalle_id"), 
        col("det.factura"), 
        col("det.producto"), 
        col("prod.producto_id"), 
        col("det.unidades"), 
        col("det.precio_unitario"), 
        col("det.Subtotal"),
        current_timestamp().alias("fecha_actualizacion")
    )
)

In [0]:
#df_detalles.write.mode("overwrite").format("delta").saveAsTable("linio.silver_detalles")

In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta

fecha = datetime.strptime(fecha_carga,"%Y-%m-%d")

particiones = [
    (fecha-relativedelta(months=2)).replace(day=1),
    (fecha-relativedelta(months=1)).replace(day=1),
    fecha.replace(day=1)
]

particiones = [
    p.strftime("%Y-%m-%d")
    for p in particiones
]

print(particiones)

In [0]:
if tipo_carga == "Historico":
    df_clientes.write.mode("overwrite").format("delta").saveAsTable("linio.silver_clientes")
    df_productos.write.mode("overwrite").format("delta").saveAsTable("linio.silver_productos")
    df_detalles.write.mode("overwrite").format("delta").saveAsTable("linio.silver_detalles")
    df_compras.write.mode("overwrite").format("delta").saveAsTable("linio.silver_compras")
else:
    from delta.tables import DeltaTable

    delta_productos = DeltaTable.forName(spark, "linio.silver_productos")
    (delta_productos.alias("prod")
    .merge(
     df_productos.alias("new"),
     "prod.producto = new.producto"
    )
    .whenMatchedUpdate(set={
     "categoria": "new.categoria",
     "subcategoria": "new.subcategoria",
     "fecha_actualizacion": "new.fecha_actualizacion"
     })
    .whenNotMatchedInsert(values={
     "producto": "new.producto",
     "categoria": "new.categoria",
     "subcategoria": "new.subcategoria",
     "fecha_actualizacion": "new.fecha_actualizacion"
    })
    .execute())

    delta_clientes = DeltaTable.forName(spark, "linio.silver_clientes")
    delta_clientes.alias("cli").merge(df_clientes.alias("nucli"), "cli.cliente_id = nucli.cliente_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    delta_detalles = DeltaTable.forName(spark, "linio.silver_detalles")
    delta_detalles.alias("det").merge(df_detalles.alias("new"), "det.detalle_id = new.detalle_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    delta_compras = DeltaTable.forName(spark, "linio.silver_compras")
    delta_compras.alias("com").merge(df_compras.alias("new"), "com.venta_id = new.venta_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    df_compras_incremental = df_compras.filter(col("periodo").isin(particiones))

    (df_compras_incremental.write
        .mode("overwrite")
        .format("delta")
        .option("replaceWhere", "periodo IN (" + ",".join([f"'{p}'" for p in particiones]) + ")")
        .partitionBy("periodo")
        .saveAsTable("linio.silver_compras"))  